#  Simple Regression with PyTorch
  Topic: Predicting Flood Water Depth from Rainfall Intensity

Goal: Train a small neural network that learns the relationship between rainfall intensity (mm/hr) and flood water depth (cm).

Steps:
- Generate synthetic data
- Build a Dataset and DataLoader
- Define a neural network model
- Train with a loss function and optimizer
- Evaluate and visualize results

In [2]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Fix random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# ── 1. Generate Synthetic Data ────────────────────────────────────────────────
#
# True relationship (unknown to the model):
#   depth = 0.8 * rainfall^1.2 + noise
#
# This mimics a nonlinear rainfall-runoff response.

N = 200                                         # number of samples
rainfall = np.random.uniform(0, 50, N)          # rainfall intensity  [mm/hr]
noise    = np.random.normal(0, 2, N)            # measurement noise   [cm]
depth    = 0.8 * rainfall**1.2 + noise          # flood water depth   [cm]

In [ ]:
# Normalize inputs to [0, 1] — always good practice!
rainfall_min, rainfall_max = rainfall.min(), rainfall.max()
rainfall_norm = (rainfall - rainfall_min) / (rainfall_max - rainfall_min)
print(f"Normalized rainfall range: {min(rainfall_norm):.2f} to {max(rainfall_norm):.2f}")

In [ ]:
# Normalize targets too (helps training stability)
depth_min, depth_max = depth.min(), depth.max()
depth_norm = (depth - depth_min) / (depth_max - depth_min)
print(f"Normalized depth range: {min(depth_norm):.2f} to {max(depth_norm):.2f}")

In [ ]:
# Train / validation / test split  (60 / 20 / 20)
# reshape to (N, 1) for PyTorch compatibility
split = int(0.6 * N)
X_train = rainfall_norm[:split].astype(np.float32).reshape(-1, 1)
y_train = depth_norm[:split].astype(np.float32).reshape(-1, 1)

X_val   = rainfall_norm[split:split+int(0.2 * N)].astype(np.float32).reshape(-1, 1)
y_val   = depth_norm[split:split+int(0.2 * N)].astype(np.float32).reshape(-1, 1)

X_test  = rainfall_norm[split+int(0.2 * N):].astype(np.float32).reshape(-1, 1)
y_test  = depth_norm[split+int(0.2 * N):].astype(np.float32).reshape(-1, 1)

print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Testing  samples   : {len(X_test)}")

In [ ]:
# ── 2. Build a PyTorch Dataset and DataLoader ─────────────────────────────────
#
# PyTorch wants data wrapped in a Dataset object so it can shuffle,
# batch, and feed samples to the model automatically.

class FloodDataset(Dataset):
    """Pairs of (rainfall, depth) observations."""

    def __init__(self, X, y):
        # Convert numpy arrays → PyTorch tensors
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)                  # total number of samples

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]    # one (input, label) pair

train_dataset = FloodDataset(X_train, y_train)
val_dataset   = FloodDataset(X_val,   y_val)
test_dataset  = FloodDataset(X_test,  y_test)

# DataLoader handles batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Number of training batches  : {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of testing batches   : {len(test_loader)}")


In [ ]:
# ── 3. Define the Neural Network ──────────────────────────────────────────────
#
# Architecture:  1 → 32 → 32 → 1

class RainfallDepthNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1, 32),    # input layer  : 1 feature → 32 neurons
            nn.ReLU(),           # activation   : adds nonlinearity
            nn.Linear(32, 32),   # hidden layer : 32 → 32 neurons
            nn.ReLU(),
            nn.Linear(32, 1),    # output layer : 32 → 1 prediction
        )

    def forward(self, x):
        return self.network(x)

model = RainfallDepthNet()
print(f"\nModel architecture:\n{model}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {sum(p.numel() for p in model.state_dict().values())}")
print(f"Total trainable parameters: {total_params}")


In [ ]:
# ── 4. Loss Function and Optimizer ────────────────────────────────────────────
#
# Loss     : Mean Squared Error — penalizes large prediction errors
# Optimizer: Adam — adapts the learning rate automatically

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# ── 5. Training Loop ──────────────────────────────────────────────────────────

EPOCHS = 200
train_losses = []
val_losses   = []

for epoch in range(1, EPOCHS + 1):

    # ── Training phase ────────────────────────────────────────────────────────
    model.train()                           # tell PyTorch we are training
    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()                   # 1. clear old gradients
        predictions = model(X_batch)            # 2. forward pass
        loss = criterion(predictions, y_batch)  # 3. compute loss
        loss.backward()                         # 4. backpropagation
        optimizer.step()                        # 5. update weights

        running_loss += loss.item() * len(X_batch)

    train_losses.append(running_loss / len(train_dataset))

    # ── Evaluation phase ──────────────────────────────────────────────────────
    model.eval()                            # tell PyTorch we are evaluating
    with torch.no_grad():                   # no gradient tracking needed
        val_loss = 0.0
        for X_batch, y_batch in val_loader:
            preds = model(X_batch)
            val_loss += criterion(preds, y_batch).item() * len(X_batch)
        val_losses.append(val_loss / len(val_dataset))

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}/{EPOCHS}  |  "
              f"Train Loss: {train_losses[-1]:.4f}  |  "
              f"Val Loss:  {val_losses[-1]:.4f}")

In [ ]:
# 

In [ ]:
# Loss curves
plt.plot(train_losses, label="Train loss", color="#1f77b4")
plt.plot(val_losses,  label="Validation loss",  color="#ff7f0e", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(alpha=0.3)


In [ ]:

# Panel B: Predictions vs ground truth
ax = axes[1]

# Generate a smooth curve for visualization
x_vis = np.linspace(0, 1, 300).astype(np.float32).reshape(-1, 1)
model.eval()
with torch.no_grad():
    y_vis = model(torch.from_numpy(x_vis)).numpy()

# Denormalize back to physical units
x_vis_phys = x_vis * (rainfall_max - rainfall_min) + rainfall_min
y_vis_phys  = y_vis * (depth_max   - depth_min)   + depth_min

X_test_phys = X_test * (rainfall_max - rainfall_min) + rainfall_min
y_test_phys = y_test * (depth_max   - depth_min)   + depth_min

ax.scatter(X_test_phys, y_test_phys, alpha=0.6,
           color="#2ca02c", label="Ground truth (test)", s=20, zorder=2)
ax.plot(x_vis_phys, y_vis_phys,
        color="#d62728", linewidth=2, label="Model prediction", zorder=3)
ax.set_xlabel("Rainfall Intensity (mm/hr)")
ax.set_ylabel("Flood Depth (cm)")
ax.set_title("Model Fit on Test Data")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("regression_results.png", dpi=150)
plt.show()
print("\nPlot saved to regression_results.png")

# ── 7. A Single Inference Example ────────────────────────────────────────────
#
# This is what a model looks like in deployment — you feed it new data
# and it returns a prediction.

new_rainfall_mmhr = 30.0                    # new observation: 30 mm/hr

# Normalize the input the same way as training data
x_new = (new_rainfall_mmhr - rainfall_min) / (rainfall_max - rainfall_min)
x_tensor = torch.tensor([[x_new]], dtype=torch.float32)

model.eval()
with torch.no_grad():
    y_pred_norm = model(x_tensor).item()

# Denormalize the output
y_pred = y_pred_norm * (depth_max - depth_min) + depth_min

print(f"\nInference example:")
print(f"  Input  : {new_rainfall_mmhr} mm/hr rainfall")
print(f"  Output : {y_pred:.1f} cm predicted flood depth")